# Lesson 04 - ਟੂਲ ਯੂਜ਼ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸੋਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ (ਪਾਇਥਨ) ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਕ੍ਰਿਤ੍ਰਿਮ ਬੁੱਧੀ ਏਜੰਟਾਂ ਲਈ **ਟੂਲ ਯੂਜ਼** ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਸਿੱਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਦੇ ਹਾਂ:

- `@tool` ਡੈਕੋਰੇਟਰ ਅਤੇ ਟਾਈਪ ਕੀਤੇ ਪੈਰامیਟਰਾਂ ਨਾਲ ਫੰਕਸ਼ਨ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ
- ਮਾਡਲ ਨੂੰ ਪਤਾ ਹੋਵੇ ਕਿ ਹਰ ਟੂਲ ਕੀ ਕਰਦਾ ਹੈ ਇਸ ਲਈ ਟੂਲ ਸਕੀਮਾਵਾਂ ਮੁਹੱਈਆ ਕਰਵਾਉਣਾ
- `approval_mode` ਨਾਲ ਟੂਲ ਅਮਲਦਾਰੀ 'ਤੇ ਨਿਯੰਤਰਣ
- ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਅਤੇ `response_format` ਰਾਹੀਂ **ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** ਵਾਪਸ ਕਰਨਾ

ਪਟਾਕਾ ਇੱਕ **ਟ੍ਰੈਵਲ ਬੁਕਿੰਗ ਏਜੰਟ** ਹੈ ਜੋ ਮੰਜ਼ਿਲਾਂ ਦੀ ਖੋਜ ਕਰ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਚੈੱਕ ਕਰ ਸਕਦਾ ਹੈ, ਅਤੇ ਉਡਾਣ ਦੀ ਜਾਣਕਾਰੀ ਹਾਸਲ ਕਰ ਸਕਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ਡੈਕੋਰੇਟਰ ਨਾਲ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ

`@tool` ਡੈਕੋਰੇਟਰ ਇੱਕ ਸਧਾਰਣ Python ਫੰਕਸ਼ਨ ਨੂੰ ਇੱਕ ਟੂਲ ਵਿੱਚ ਬਦਲ ਦਿੰਦਾ ਹੈ ਜਿਸਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
ਮੁੱਖ ਬਿੰਦੂ:

- **ਡੌਕਸਟਰਿੰਗ** ਮਾਡਲ ਨੂੰ ਵੇਖਣ ਲਈ ਟੂਲ ਦਾ ਵੇਰਵਾ ਬਣ ਜਾਂਦੀ ਹੈ।
- **ਟਾਈਪ ਐਨੋਟੇਸ਼ਨ** (ਵਰਣਨ ਸਮੇਤ `Annotated` ਸਮੇਤ) ਟੂਲ ਸਕੀਮਾ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ।
- `approval_mode` ਨਿਯੰਤਰਿਤ ਕਰਦਾ ਹੈ ਕਿ ਯੂਜ਼ਰ ਨੂੰ ਹਰ ਕਾਲ ਨਾਲ ਪਹਿਲਾਂ ਮਨਜ਼ੂਰੀ ਦੇਣੀ ਚਾਹੀਦੀ ਹੈ ਜਾਂ ਨਹੀਂ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ਇਕ ਏਜੰਟ ਬਣਾਉਣਾ ਜਿਸ ਵਿੱਚ ਕਈ ਟੂਲ ਹਨ

ਤਿੰਨੋ ਟੂਲਾਂ ਨੂੰ ਕਲਾਇੰਟ ਨੂੰ ਦਿਓ ਤਾਂ ਜੋ ਮਾਡਲ ਉਸ ਅਨੁਸਾਰ ਜਿਹੜੇ ਜ਼ਰੂਰੀ ਹੋਣ ਉਹਨਾਂ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰ ਸਕੇ ਅਤੇ ਯੂਜ਼ਰ ਦੇ ਸਵਾਲ ਦਾ ਜਵਾਬ ਦੇ ਸਕੇ।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ਸਾਧਨਾਂ ਨਾਲ ਬਣਾਇਆ ਗਿਆ ਸੰਰਚਿਤ ਨਤੀਜਾ

`response_format` ਨੂੰ ਇੱਕ Pydantic ਮਾਡਲ ਤੇ ਸੈੱਟ ਕਰ ਕੇ, ਏਜੰਟ ਨੂੰ ਮੁਕਤ-ਰੂਪ ਰੂਪ ਵਿੱਚ ਲਿਖਿਆ ਗਇਆ ਟੈਕਸਟ ਵਜੋਂ ਨਹੀਂ, ਸਗੋਂ ਚੰਗੀ ਤਰ੍ਹਾਂ ਟਾਈਪ ਕੀਤਾ ਗਿਆ JSON ਆਬਜੈਕਟ ਵਾਪਸ ਕਰਨ ਲਈ ਮਜ਼ਬੂਰ ਕੀਤਾ ਜਾਂਦਾ ਹੈ। ਇਹ ਉਸ ਵੇਲੇ ਲਾਭਦਾਇਕ ਹੁੰਦਾ ਹੈ ਜਦੋਂ ਡਾਊਨਸਟਰੀਮ ਕੋਡ ਨੂੰ ਨਤੀਜੇ ਨੂੰ ਪ੍ਰੋਗ੍ਰਾਮੈਟਿਕ ਤੌਰ 'ਤੇ ਖਪਤ ਕਰਨ ਦੀ ਲੋੜ ਹੋਵੇ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ਟੂਲ ਮਨਜ਼ੂਰੀ ਦੇ ਨਮੂਨੇ

`@tool` ਤੇ `approval_mode` ਪੈਰਾਮੀਟਰ ਇਹ ਨਿਰਧਾਰਿਤ ਕਰਦਾ ਹੈ ਕਿ ਕੀ ਟੂਲ ਕਾਲਾਂ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਮਨੁੱਖੀ ਮਨਜ਼ੂਰੀ ਦੀ ਲੋੜ ਹੈ:

| ਮੋਡ | ਵਿਵਹਾਰ |
|---|---|
| `"never_require"` | ਟੂਲ ਆਪੋ-ਆਪ ਚਲਦਾ ਹੈ — ਕਿਸੇ ਉਪਭੋਗਤਾ ਦੀ ਪੁਸ਼ਟੀ ਦੀ ਲੋੜ ਨਹੀਂ। |
| `"always_require"` | ਹਰ ਕਾਲ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਉਪਭੋਗਤਾ ਵੱਲੋਂ ਮਨਜ਼ੂਰ ਕੀਤਾ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। |

ਆਮ ਤੌਰ ਤੇ ਉਹਨਾਂ ਟੂਲਾਂ ਲਈ `"always_require"` ਦੀ ਵਰਤੋਂ ਕਰੋ ਜਿਨ੍ਹਾਂ ਦੇ ਸਾਈਡ-ਇਫੈਕਟ ਹੁੰਦੇ ਹਨ (ਜਿਵੇਂ ਕਿ ਟਿਕਟ ਬੁਕਿੰਗ, ਕਰੈਡਿਟ ਕਾਰਡ ਚਾਰਜ ਕਰਨਾ) ਤਾਂ ਜੋ ਮਨੁੱਖ ਲੂਪ ਵਿੱਚ ਰਹੇ।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

ਇਸ ਪਾਠ ਵਿਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ:

1. **ਉਪਕਰਨਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ** `@tool` ਡੀਕੋਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਟਾਈਪ ਕੀਤੇ ਪੈਰਾਮੀਟਰਾਂ ਅਤੇ ਡੌਕਸਟ੍ਰਿੰਗਜ਼ ਨਾਲ ਜੋ ਟੂਲ ਸਕੀਮਾ ਵਜੋਂ ਕੰਮ ਕਰਦੀਆਂ ਹਨ।
2. **ਕਈ ਉਪਕਰਨਾਂ ਨੂੰ ਜੋੜਨਾ** ਤਾਂ ਜੋ ਏਜੰਟ ਉਹਨਾਂ ਨੂੰ ਅਨੁਕ੍ਰਮ ਵਿੱਚ ਕਾਲ ਕਰ ਸਕੇ ਅਤੇ ਜਟਿਲ ਪ੍ਰਸ਼ਨਾਂ ਦੇ ਜਵਾਬ ਦੇ ਸਕੇ।
3. **ਸੰਰਚਿਤ ਨਿਕਾਸ ਵਾਪਸ ਕਰਨਾ** ਇੱਕ Pydantic ਮਾਡਲ ਨੂੰ `response_format` ਵਜੋਂ ਪਾਸ ਕਰਕੇ।
4. **ਉਪਕਰਨ ਮਨਜ਼ੂਰੀ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਨਾ** `approval_mode` ਨਾਲ ਤਾਂ ਜੋ ਸੰਵੇਦਨਸ਼ੀਲ ਕਾਰਵਾਈਆਂ ਲਈ ਇਨਸਾਨ ਨੂੰ ਲੂਪ ਵਿੱਚ ਰੱਖਿਆ ਜਾ ਸਕੇ।

ਇਹ ਨਮੂਨੇ ਭਰੋਸੇਮੰਦ, ਉਤਪਾਦਨ-ਤਿਆਰ ਏਜੰਟ ਬਣਾਉਣ ਦੀ ਬੁਨਿਆਦ ਧਾਰਦੇ ਹਨ ਜੋ ਬਾਹਰੀ ਪ੍ਰਣਾਲੀਆਂ ਨਾਲ ਸੁਰੱਖਿਅਤ ਤਰੀਕੇ ਨਾਲ ਇੰਟਰਐਕਟ ਕਰ ਸਕਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰਤਾ**:  
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਇਸ ਗੱਲ ਦਾ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸੁਚਾਲਿਤ ਅਨੁਵਾਦ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਪਸ਼ਟਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਉਸ ਦੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਅਹਮ ਜਾਣਕਾਰੀ ਲਈ ਪ੍ਰੋਫੈਸ਼ਨਲ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੇ ਇਸਤੇਮਾਲ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਅਸੀਂ ਜਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
